In [ ]:
!uv pip install torch==2.8.0 torchvision==0.23.0 nvidia-modelopt git+https://github.com/ultralytics/ultralytics@qat-nvidia

Using Python 3.12.13 environment at: /usr
Resolved 65 packages in 6.98s
Prepared 8 packages in 21.47s
Uninstalled 4 packages in 830ms
Installed 8 packages in 353ms
 + ninja==1.13.0
 + nvidia-modelopt==0.42.0
 - nvidia-nccl-cu12==2.27.5
 + nvidia-nccl-cu12==2.27.3
 - torch==2.10.0+cu128
 + torch==2.8.0
 - torchvision==0.25.0+cu128
 + torchvision==0.23.0
 - triton==3.6.0
 + triton==3.4.0
 + ultralytics==8.4.8 (from git+https://github.com/ultralytics/ultralytics@e80696dd7a09a48d035501e8145f28ffe0f18b37)
 + ultralytics-thop==2.0.18


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import torch
import math
from ultralytics import YOLO
from ultralytics.utils.torch_utils import ModelEMA
from ultralytics.utils import LOGGER
import modelopt.torch.prune as mtp

teacher_path = '/content/drive/MyDrive/PBL5_Final_Results/v8m_eb_observer5/weights/best.pt'
data_yaml = '/content/drive/MyDrive/Work3.yolov8/data.yaml'

model = YOLO("yolov8n.pt")

# Load teacher một lần duy nhất, không train
teacher = YOLO(teacher_path).model.eval()
for p in teacher.parameters():
    p.requires_grad = False

class PrunedTrainer(model.task_map[model.task]["trainer"]):
    def _setup_train(self):
        super()._setup_train()

        # Đưa teacher lên đúng device sau khi trainer biết device
        self._teacher = teacher.to(self.device)

        def collect_func(batch):
            return self.preprocess_batch(batch)["img"]

        def score_func(m):
            m.eval()
            self.validator.args.save = False
            self.validator.args.plots = False
            self.validator.args.verbose = False
            self.validator.args.data = data_yaml
            metrics = self.validator(model=m)
            return metrics["fitness"]

        prune_constraints = {"flops": "70%"}
        self.model.is_fused = lambda: True
        LOGGER.info("--- ĐANG TÌM CẤU TRÚC TỐI ƯU (FASTNAS) ---")

        self.model, prune_res = mtp.prune(
            model=self.model,
            mode="fastnas",
            constraints=prune_constraints,
            dummy_input=torch.randn(1, 3, self.args.imgsz, self.args.imgsz).to(self.device),
            config={
                "score_func": score_func,
                "checkpoint": "modelopt_fastnas_search_checkpoint.pth",
                "data_loader": self.train_loader,
                "collect_func": collect_func,
                "max_iter_data_loader": 20,
            },
        )

        self.model.to(self.device)
        self.ema = ModelEMA(self.model)

        weight_decay = self.args.weight_decay * self.batch_size * self.accumulate / self.args.nbs
        iterations = math.ceil(len(self.train_loader.dataset) / max(self.batch_size, self.args.nbs)) * self.epochs
        self.optimizer = self.build_optimizer(
            model=self.model,
            name=self.args.optimizer,
            lr=self.args.lr0,
            momentum=self.args.momentum,
            decay=weight_decay,
            iterations=iterations,
        )
        self._setup_scheduler()
        LOGGER.info(f"--- ĐÃ CẮT VẬT LÝ THÀNH CÔNG: {prune_res} ---")

    def loss(self, batch, preds=None):
        """Override loss để thêm KD loss từ teacher"""
        # Loss gốc của student
        student_loss, student_loss_items = super().loss(batch, preds)

        # KD loss: student học từ teacher
        imgs = batch["img"].to(self.device)
        with torch.no_grad():
            teacher_preds = self._teacher(imgs)

        # Forward student để lấy predictions cùng scale
        student_preds = self.model(imgs)

        # Tính KD loss trên feature maps (MSE giữa student và teacher)
        kd_loss = 0
        if isinstance(teacher_preds, (list, tuple)) and isinstance(student_preds, (list, tuple)):
            for t_feat, s_feat in zip(teacher_preds, student_preds):
                if isinstance(t_feat, torch.Tensor) and isinstance(s_feat, torch.Tensor):
                    # Nếu shape khác nhau (do prune), dùng interpolate
                    if t_feat.shape != s_feat.shape:
                        s_feat = torch.nn.functional.interpolate(
                            s_feat, size=t_feat.shape[-2:], mode='bilinear', align_corners=False
                        )
                    kd_loss += torch.nn.functional.mse_loss(s_feat, t_feat)

        kd_weight = 0.5  # Điều chỉnh tùy mAP recover
        total_loss = student_loss + kd_weight * kd_loss

        return total_loss, student_loss_items


model.train(
    data=data_yaml,
    trainer=PrunedTrainer,
    epochs=70,
    imgsz=640,
    batch=32,
    lr0=0.0001,
    exist_ok=True,
    warmup_epochs=0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.3,
    degrees=5.0,
    perspective=0.0003,
    device=0,
    project='/content/drive/MyDrive/PBL5_Final_Results',
    name='26n_ModelOpt_Physical_Pruning_KD'
)

AttributeError: partially initialized module 'torch' has no attribute 'nn' (most likely due to a circular import)

In [ ]:
from ultralytics import YOLO

# 1. Load mô hình bạn muốn đánh giá (Nano đã Prune hoặc Teacher Medium)
model_path = '/content/drive/MyDrive/PBL5_Final_Results/v8n_ModelOpt_Physical_Pruning_KD/weights/best.pt'
model = YOLO(model_path)

# 2. Chạy Validation
# data: Đường dẫn đến file data.yaml của bạn
# split: 'val' hoặc 'test' (tùy vào tập bạn muốn đánh giá)
# imgsz: Phải khớp với imgsz lúc bạn train (thường là 640)
# batch: Có thể để cao hơn lúc train vì không tốn Gradient
metrics = model.val(
    data='/content/drive/MyDrive/Work3.yolov8/data.yaml',
    split='test',
    imgsz=640,
    batch=32,
    conf=0.001, # Ngưỡng tự tin thấp để vẽ đường cong PR đầy đủ
    iou=0.6,    # Ngưỡng giao thoa
    device=0,   # Sử dụng GPU
    save_json=True # Lưu file json để nếu cần tính thêm metric khác
)

# 3. In các thông số quan trọng nhất
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

Ultralytics 8.4.8 🚀 Python-3.12.13 torch-2.8.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 36.2±6.6 MB/s, size: 82.1 KB)
val: Scanning /content/drive/MyDrive/Work3.yolov8/test/labels.cache... 513 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 513/513 126.6Mit/s 0.0s
val: /content/drive/MyDrive/Work3.yolov8/test/images/test__20260320_100648_frame_000046_jpg.rf.2c56ff5ed1ea9ce06ce276fbc56c99a4.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Work3.yolov8/test/images/train__20260227_165904_frame_000513_jpg.rf.10b46e141216861a23927753232e919d.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Work3.yolov8/test/images/train__20260314_221808_frame_000028_jpg.rf.3ec371a42d8058529925d0f68482cd04.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Work3.yolov8/test/images/valid__20260320_101057_frame_000329_jpg.rf.09397d1ef7c39950625d37cb93a5b4be.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Work3.yolov8/test

In [ ]:
orig_model = YOLO('yolov8n.pt')

In [ ]:
orig_model.info()

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8575488)

In [ ]:
pruned_model = YOLO("/content/drive/MyDrive/PBL5_Final_Results/v8n_ModelOpt_Physical_Pruning_KD/weights/best.pt")

In [ ]:
pruned_model.info()

Model summary: 130 layers, 2,036,268 parameters, 2,036,268 gradients, 5.7 GFLOPs


(130, 2036268, 2036268, 5.7181695999999995)

In [ ]:
!pip uninstall torchprofile -y
!pip install torchprofile==0.0.4

In [ ]:
from ultralytics import YOLO

# 1. Load model sau khi đã Prune và Fine-tune xong
model_path = '/content/drive/MyDrive/PBL5_Final_Results/v8n_ModelOpt_Physical_Pruning_KD/weights/best.pt'
model = YOLO(model_path)

# 2. Export sang ONNX
# imgsz: Nên để 640 (khớp với lúc train)
# dynamic: False (Jetson Nano chạy ổn định nhất với Fixed Shape)
# simplify: True (Rút gọn đồ thị toán học, cực kỳ quan trọng cho TensorRT)
success = model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=12  # Opset 12 thường tương thích tốt nhất với TensorRT trên JetPack 4.6/5.x
)

print(f"Export thành công: {success}")

Ultralytics 8.4.8 🚀 Python-3.12.13 torch-2.8.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/MyDrive/PBL5_Final_Results/v8n_ModelOpt_Physical_Pruning_KD/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (7.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 361ms
Prepared 4 packages in 1.86s
Installed 4 packages in 447ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.24.4
 + onnxslim==0.1.90

requirements: AutoUpdate success ✅ 3.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 12...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export su

In [ ]:
from ultralytics import YOLO

# 1. Load mô hình bạn muốn đánh giá (Nano đã Prune hoặc Teacher Medium)
model_path = '/content/drive/MyDrive/PBL5_Final_Results/v8n_ModelOpt_Physical_Pruning_KD/weights/best.pt'
model = YOLO(model_path)

# 2. Chạy Validation
# data: Đường dẫn đến file data.yaml của bạn
# split: 'val' hoặc 'test' (tùy vào tập bạn muốn đánh giá)
# imgsz: Phải khớp với imgsz lúc bạn train (thường là 640)
# batch: Có thể để cao hơn lúc train vì không tốn Gradient
metrics = model.val(
    data='/content/drive/MyDrive/Work3.yolov8/data.yaml',
    split='val',
    imgsz=640,
    batch=32,
    conf=0.001, # Ngưỡng tự tin thấp để vẽ đường cong PR đầy đủ
    iou=0.6,    # Ngưỡng giao thoa
    device=0,   # Sử dụng GPU
    save_json=True # Lưu file json để nếu cần tính thêm metric khác
)

# 3. In các thông số quan trọng nhất
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")